# GxP-LLM Quantization: GPTQ, AWQ, FP8

Quantize the merged 16-bit model. Run on Kaggle (GPTQ/AWQ) or Modal (FP8 on H100).

In [ ]:
# Install dependencies
%pip install -q torch==2.5.1 transformers==4.46.3 accelerate==0.34.2
%pip install -q auto-gptq==0.7.1 autoawq==0.2.7

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Config
MODEL_PATH = "/kaggle/input/gxp-model/merged_16bit"  # or local path
OUTPUT_DIR = "./quantized"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# GPTQ Quantization (4-bit, group_size=128)
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

quantize_config = BaseQuantizeConfig(
    bits=4,
    group_size=128,
    desc_act=False,
    sym=True,
)

model = AutoGPTQForCausalLM.from_pretrained(
    MODEL_PATH,
    quantize_config=quantize_config,
    device_map="auto",
    trust_remote_code=True,
)

# Calibration data (use subset of train)
import json
with open("/kaggle/input/gxp-data/train.jsonl") as f:
    calib_data = [json.loads(line) for line in f][:128]

calib_texts = []
for ex in calib_data:
    text = ex["messages"][0]["content"] + "\n" + ex["messages"][1]["content"]
    calib_texts.append(text)

model.quantize(calib_texts)
model.save_quantized(f"{OUTPUT_DIR}/gptq-4bit")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/gptq-4bit")

print("GPTQ done: saved to ./quantized/gptq-4bit")

In [ ]:
# AWQ Quantization (4-bit, group_size=128)
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

model = AutoAWQForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
)

# Calibration
calib_texts = []
with open("/kaggle/input/gxp-data/train.jsonl") as f:
    for i, line in enumerate(f):
        if i >= 128:
            break
        ex = json.loads(line)
        calib_texts.append(ex["messages"][0]["content"] + "\n" + ex["messages"][1]["content"])

model.quantize(tokenizer, quant_config=quant_config, calib_data=calib_texts)
model.save_quantized(f"{OUTPUT_DIR}/awq-4bit")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/awq-4bit")

print("AWQ done: saved to ./quantized/awq-4bit")

In [ ]:
# FP8 Quantization (requires H100 - run on Modal, not Kaggle T4)
# See quantize/fp8_quantize.py for Modal deployment
print("FP8 quantization requires H100 - run on Modal (see quantize/fp8_quantize.py)")

In [ ]:
# Quick perplexity check on each quantized model
from transformers import AutoModelForCausalLM
import torch

def quick_perplexity(model_path, tokenizer, n_samples=10):
    model = AutoModelForCausalLM.from_pretrained(
        model_path, device_map="auto", trust_remote_code=True, torch_dtype=torch.float16
    )
    model.eval()

    with open("/kaggle/input/gxp-data/eval.jsonl") as f:
        eval_data = [json.loads(line) for line in f][:n_samples]

    total_loss = 0
    for ex in eval_data:
        text = ex["messages"][0]["content"] + "\n" + ex["messages"][1]["content"] + "\n" + ex["messages"][2]["content"]
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            total_loss += outputs.loss.item()
    
    avg_loss = total_loss / len(eval_data)
    ppl = torch.exp(torch.tensor(avg_loss)).item()
    return ppl

for name in ["gptq-4bit", "awq-4bit"]:
    path = f"{OUTPUT_DIR}/{name}"
    try:
        ppl = quick_perplexity(path, tokenizer)
        print(f"{name}: perplexity = {ppl:.2f}")
    except Exception as e:
        print(f"{name}: error - {e}")

In [ ]:
# Upload quantized models to W&B
import wandb
import os

wandb.login(key=os.environ.get("WANDB_API_KEY"))
wandb.init(project="gxp-llm", name="quantization", job_type="quantize")

for name in ["gptq-4bit", "awq-4bit"]:
    path = f"{OUTPUT_DIR}/{name}"
    if os.path.exists(path):
        artifact = wandb.Artifact(f"qwen2.5-7b-gxp-{name}", type="model")
        artifact.add_dir(path)
        wandb.log_artifact(artifact)

wandb.finish()